In [1]:
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, confusion_matrix # Added import

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the dataset
dataset = load_dataset("ailsntua/QEvasion")

# Prepare labels
labels = dataset['train'].unique('clarity_label')
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

def add_labels(example):
    example['labels'] = label2id[example['clarity_label']]
    return example

dataset = dataset.map(add_labels)
dataset = dataset.remove_columns([
    col for col in dataset['train'].column_names if col not in ['question', 'interview_answer', 'labels']
])

print("Dataset ready:")
print(dataset)
print(f"Labels mapped: {label2id}")


# Calculate class weights for imbalanced data
def get_class_weights(dataset, num_labels):
    label_counts = Counter(dataset["train"]["labels"])
    total_samples = len(dataset["train"])

    # Calculate class weights (inverse frequency)
    class_weights = []
    for i in range(num_labels):
        count = label_counts.get(i, 1)  # avoid division by zero
        weight = total_samples / (num_labels * count)
        class_weights.append(weight)

    # Move the tensor to the active device (GPU)
    return torch.tensor(
        class_weights, dtype=torch.float32, device=device
    )


# Get class weights
class_weights = get_class_weights(dataset, num_labels)
print(f"Class weights: {class_weights}")
print(f"Class weights device: {class_weights.device}")  # Verify it's on CUDA

# Focal Loss implementation :cite[1]:cite[8]
class FocalLoss(nn.Module):
    """
    Multi-class Focal loss implementation
    Focal loss helps address class imbalance by focusing on hard examples

    Args:
        gamma (float): Focusing parameter, higher values put more focus on hard examples
        weight (Tensor): Class weights tensor for handling imbalanced data
        ignore_index (int): Index to ignore in loss calculation
    """
    def __init__(self, gamma=1.0, weight=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, input, target):
        # Calculate cross entropy loss
        ce_loss = F.cross_entropy(input, target, reduction='none', weight=self.weight, ignore_index=self.ignore_index)

        # Get probabilities
        pt = torch.exp(-ce_loss)

        # Compute focal loss
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()


# Updated Custom Trainer with corrected compute_loss signature
class CustomTrainer(Trainer):
    """
    Custom trainer that uses Focal Loss with class weights
    This subclass overrides the compute_loss method to use our custom loss function
    """

    def __init__(self, *args, class_weights=None, focal_gamma=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_loss = FocalLoss(gamma=focal_gamma, weight=class_weights)

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        # Extract labels and run model forward pass
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Compute focal loss with class weights
        loss = self.focal_loss(logits, labels)

        # Handle return_outputs as required by the Trainer
        return (loss, outputs) if return_outputs else loss

# Tokenization and model setup
model_checkpoint = "answerdotai/ModernBERT-large"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(
        examples['question'],
        examples['interview_answer'],
        truncation=True,
        padding="max_length",
        max_length=1680
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    # Use macro averaging for balanced metrics across classes
    precision_macro = precision_score(labels, predictions, average='macro', zero_division=0)
    recall_macro = recall_score(labels, predictions, average='macro', zero_division=0)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)

    # Keep weighted for comparison
    precision_weighted = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall_weighted = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1_weighted = f1_score(labels, predictions, average='weighted', zero_division=0)

    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'precision_weighted': precision_weighted,
        'recall_macro': recall_macro,
        'recall_weighted': recall_weighted
    }

# --- MODIFICATION 1: Updated TrainingArguments ---
# We now evaluate, save, and load the best model based on 'f1_macro'
training_args = TrainingArguments(
    output_dir="ModernBERT_QEvasion_model",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    num_train_epochs=8,
    weight_decay=0.01,
    eval_strategy="epoch",          # <--- MODIFIED (was "no")
    save_strategy="epoch",
    load_best_model_at_end=True,    # <--- MODIFIED (was False)
    metric_for_best_model="f1_macro", # <--- NEW
    greater_is_better=True,         # <--- NEW
    push_to_hub=False,
    logging_steps=100,
    report_to="none",
    fp16=True,  # Enable mixed precision (reduces memory usage)
    gradient_checkpointing=True,
)

# --- MODIFICATION 2: Updated CustomTrainer instantiation ---
# We pass the test set to eval_dataset
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],  # <--- NEW
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,  # Pass the calculated class weights
    focal_gamma=1.0,  # You can adjust this parameter
)

print(f"Using class weights: {class_weights}")
print("Starting training with Focal Loss (evaluating on test set after each epoch)...")
trainer.train()

print("Training completed!")

# --- MODIFICATION 3: Updated Final Evaluation ---
# This will now evaluate the *best* model saved during training
# (because of load_best_model_at_end=True)
test_results = trainer.evaluate() # <--- MODIFIED (no arg needed)
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (from best epoch: {trainer.state.best_model_checkpoint})") # <--- MODIFIED
print("="*60)
for key, value in test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# Optional: Get detailed predictions
# This will also use the best model
print("\nDetailed predictions analysis (from best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                          target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

...

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2805316220.py:98: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Using class weights: tensor([1.0925, 0.5634, 3.2285], device='cuda:0')
Starting training with Focal Loss (evaluating on test set after each epoch)...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted
1,0.613200,0.469193,0.340909,0.387791,0.254102,0.518731,0.626869,0.563190,0.340909
2,0.465200,0.423000,0.600649,0.549030,0.614612,0.525594,0.652666,0.609530,0.600649
3,0.242200,0.441608,0.620130,0.569107,0.631395,0.541763,0.660027,0.629512,0.620130
4,0.157700,0.608049,0.714286,0.608010,0.697436,0.627293,0.705016,0.621943,0.714286
5,0.127100,0.907600,0.740260,0.645844,0.729658,0.671435,0.727823,0.629949,0.740260
6,0.026900,1.382408,0.753247,0.664875,0.741151,0.695838,0.743203,0.649296,0.753247
7,0.035100,1.632436,0.746753,0.667499,0.743418,0.700880,0.742759,0.643722,0.746753
8,0.003400,1.504870,0.746753,0.672021,0.743434,0.694485,0.741779,0.653995,0.746753


Training completed!



FINAL TEST RESULTS (from best epoch: ModernBERT_QEvasion_model/checkpoint-3448)
eval_loss: 1.5049
eval_accuracy: 0.7468
eval_f1_macro: 0.6720
eval_f1_weighted: 0.7434
eval_precision_macro: 0.6945
eval_precision_weighted: 0.7418
eval_recall_macro: 0.6540
eval_recall_weighted: 0.7468

Detailed predictions analysis (from best model):

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.59      0.56      0.58        79
     Ambivalent       0.80      0.84      0.82       206
Clear Non-Reply       0.68      0.57      0.62        23

       accuracy                           0.75       308
      macro avg       0.69      0.65      0.67       308
   weighted avg       0.74      0.75      0.74       308


Confusion Matrix:
[[ 44  33   2]
 [ 29 173   4]
 [  1   9  13]]


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Save the best model
drive_model_path = "/content/drive/MyDrive/ModernBERT_QEvasion_best_model_focal_newgamma"

print("=" * 60)
print("SAVING BEST MODEL TO GOOGLE DRIVE")
print("=" * 60)
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best F1 macro: {trainer.state.best_metric:.4f}")

# Save model and tokenizer
trainer.save_model(drive_model_path)
tokenizer.save_pretrained(drive_model_path)

print(f"\nModel saved to: {drive_model_path}")
print("Done!")

# Optional: list saved files
import os
if os.path.exists(drive_model_path):
    print(f"\nSaved files in {drive_model_path}:")
    for file in os.listdir(drive_model_path):
        file_path = os.path.join(drive_model_path, file)
        size = os.path.getsize(file_path) / (1024*1024)  # Size in MB
        print(f"  - {file} ({size:.1f} MB)")

Mounted at /content/drive
SAVING BEST MODEL TO GOOGLE DRIVE
Best checkpoint: ModernBERT_QEvasion_model/checkpoint-3448
Best F1 macro: 0.6720

Model saved to: /content/drive/MyDrive/ModernBERT_QEvasion_best_model_focal_newgamma
Done!

Saved files in /content/drive/MyDrive/ModernBERT_QEvasion_best_model_focal_newgamma:
  - config.json (0.0 MB)
  - model.safetensors (1510.0 MB)
  - tokenizer_config.json (0.0 MB)
  - special_tokens_map.json (0.0 MB)
  - tokenizer.json (3.4 MB)
  - training_args.bin (0.0 MB)


In [4]:
# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 13
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))



STARTING PART 2: Resuming training for 5 additional epochs...

Updated total number of epochs to: 13


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted
9,0.025200,1.326988,0.737013,0.672545,0.736107,0.670547,0.735378,0.674890,0.737013
10,0.014800,1.690260,0.746753,0.644484,0.740137,0.691386,0.739173,0.615371,0.746753
11,0.018000,1.373697,0.746753,0.677504,0.745509,0.673835,0.744806,0.682346,0.746753
12,0.007400,1.644482,0.743506,0.659228,0.741706,0.694130,0.742917,0.637033,0.743506
13,0.007100,1.673941,0.746753,0.660763,0.742409,0.709094,0.742931,0.630847,0.746753


Additional 5 epochs of training completed!

FINAL TEST RESULTS (after 13 total epochs)
(Best model from all runs: ModernBERT_QEvasion_model/checkpoint-4741)


eval_loss: 1.3737
eval_accuracy: 0.7468
eval_f1_macro: 0.6775
eval_f1_weighted: 0.7455
eval_precision_macro: 0.6738
eval_precision_weighted: 0.7448
eval_recall_macro: 0.6823
eval_recall_weighted: 0.7468

Detailed predictions analysis (from new best model):

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.61      0.57      0.59        79
     Ambivalent       0.81      0.83      0.82       206
Clear Non-Reply       0.60      0.65      0.62        23

       accuracy                           0.75       308
      macro avg       0.67      0.68      0.68       308
   weighted avg       0.74      0.75      0.75       308


Confusion Matrix:
[[ 45  31   3]
 [ 29 170   7]
 [  0   8  15]]


In [5]:
# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for even more 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 18
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))



STARTING PART 2: Resuming training for even more 5 additional epochs...

Updated total number of epochs to: 18


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted
14,0.031100,1.422427,0.750000,0.674996,0.745792,0.677474,0.744373,0.676160,0.750000
15,0.007200,1.687048,0.733766,0.639294,0.728973,0.692780,0.730041,0.608899,0.733766
16,0.005700,1.673501,0.756494,0.667206,0.751824,0.690907,0.749703,0.648576,0.756494
17,0.005900,1.611557,0.756494,0.682363,0.755756,0.694872,0.755498,0.671856,0.756494
18,0.003500,1.623231,0.756494,0.682363,0.755756,0.694872,0.755498,0.671856,0.756494


Additional 5 epochs of training completed!

FINAL TEST RESULTS (after 18 total epochs)
(Best model from all runs: ModernBERT_QEvasion_model/checkpoint-7327)


eval_loss: 1.6116
eval_accuracy: 0.7565
eval_f1_macro: 0.6824
eval_f1_weighted: 0.7558
eval_precision_macro: 0.6949
eval_precision_weighted: 0.7555
eval_recall_macro: 0.6719
eval_recall_weighted: 0.7565

Detailed predictions analysis (from new best model):

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.61      0.62      0.62        79
     Ambivalent       0.82      0.83      0.83       206
Clear Non-Reply       0.65      0.57      0.60        23

       accuracy                           0.76       308
      macro avg       0.69      0.67      0.68       308
   weighted avg       0.76      0.76      0.76       308


Confusion Matrix:
[[ 49  28   2]
 [ 30 171   5]
 [  1   9  13]]


In [6]:
# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for more more even more more 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 25
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))



STARTING PART 2: Resuming training for more more even more more 5 additional epochs...

Updated total number of epochs to: 25


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted
19,0.019700,1.568738,0.724026,0.646186,0.726517,0.640148,0.729768,0.653073,0.724026
20,0.031300,1.612822,0.714286,0.657212,0.717036,0.662969,0.720862,0.653290,0.714286
21,0.009100,1.812751,0.746753,0.687057,0.746100,0.692016,0.745533,0.682346,0.746753
22,0.009900,1.826324,0.717532,0.653079,0.720429,0.662378,0.725105,0.647236,0.717532
23,0.010900,1.777939,0.714286,0.645348,0.716376,0.665251,0.720966,0.632743,0.714286
24,0.007100,1.816614,0.720779,0.659330,0.723037,0.674757,0.727142,0.648854,0.720779
25,0.007700,1.825601,0.724026,0.650251,0.724196,0.671752,0.725959,0.634996,0.724026


Additional 5 epochs of training completed!

FINAL TEST RESULTS (after 25 total epochs)
(Best model from all runs: ModernBERT_QEvasion_model/checkpoint-9051)


eval_loss: 1.8128
eval_accuracy: 0.7468
eval_f1_macro: 0.6871
eval_f1_weighted: 0.7461
eval_precision_macro: 0.6920
eval_precision_weighted: 0.7455
eval_recall_macro: 0.6823
eval_recall_weighted: 0.7468

Detailed predictions analysis (from new best model):

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.58      0.57      0.57        79
     Ambivalent       0.82      0.83      0.82       206
Clear Non-Reply       0.68      0.65      0.67        23

       accuracy                           0.75       308
      macro avg       0.69      0.68      0.69       308
   weighted avg       0.75      0.75      0.75       308


Confusion Matrix:
[[ 45  31   3]
 [ 32 170   4]
 [  1   7  15]]


In [7]:
# these are the final additional 5 epochs after this im DONE

# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for more more even more more 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 30
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))



STARTING PART 2: Resuming training for more more even more more 5 additional epochs...

Updated total number of epochs to: 30


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted
26,0.011700,1.630481,0.733766,0.670590,0.735754,0.685650,0.739627,0.660529,0.733766
27,0.006000,1.852312,0.743506,0.661166,0.742022,0.695491,0.743588,0.639634,0.743506
28,0.003800,1.866106,0.740260,0.650484,0.739115,0.687851,0.742350,0.630344,0.740260
29,0.002400,1.814771,0.750000,0.684474,0.749209,0.709404,0.749954,0.666018,0.750000
30,0.002200,1.827739,0.743506,0.672878,0.742219,0.692004,0.741879,0.657580,0.743506


Additional 5 epochs of training completed!

FINAL TEST RESULTS (after 30 total epochs)
(Best model from all runs: ModernBERT_QEvasion_model/checkpoint-9051)


eval_loss: 1.8128
eval_accuracy: 0.7468
eval_f1_macro: 0.6871
eval_f1_weighted: 0.7461
eval_precision_macro: 0.6920
eval_precision_weighted: 0.7455
eval_recall_macro: 0.6823
eval_recall_weighted: 0.7468

Detailed predictions analysis (from new best model):

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.58      0.57      0.57        79
     Ambivalent       0.82      0.83      0.82       206
Clear Non-Reply       0.68      0.65      0.67        23

       accuracy                           0.75       308
      macro avg       0.69      0.68      0.69       308
   weighted avg       0.75      0.75      0.75       308


Confusion Matrix:
[[ 45  31   3]
 [ 32 170   4]
 [  1   7  15]]


In [8]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Save the best model
drive_model_path = "/content/drive/MyDrive/ModernBERT_QEvasion_best_model_focal_newgamma_69"

print("=" * 60)
print("SAVING BEST MODEL TO GOOGLE DRIVE")
print("=" * 60)
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best F1 macro: {trainer.state.best_metric:.4f}")

# Save model and tokenizer
trainer.save_model(drive_model_path)
tokenizer.save_pretrained(drive_model_path)

print(f"\nModel saved to: {drive_model_path}")
print("Done!")

# Optional: list saved files
import os
if os.path.exists(drive_model_path):
    print(f"\nSaved files in {drive_model_path}:")
    for file in os.listdir(drive_model_path):
        file_path = os.path.join(drive_model_path, file)
        size = os.path.getsize(file_path) / (1024*1024)  # Size in MB
        print(f"  - {file} ({size:.1f} MB)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SAVING BEST MODEL TO GOOGLE DRIVE
Best checkpoint: ModernBERT_QEvasion_model/checkpoint-9051
Best F1 macro: 0.6871

Model saved to: /content/drive/MyDrive/ModernBERT_QEvasion_best_model_focal_newgamma_69
Done!

Saved files in /content/drive/MyDrive/ModernBERT_QEvasion_best_model_focal_newgamma_69:
  - config.json (0.0 MB)
  - model.safetensors (1510.0 MB)
  - tokenizer_config.json (0.0 MB)
  - special_tokens_map.json (0.0 MB)
  - tokenizer.json (3.4 MB)
  - training_args.bin (0.0 MB)
